# Text Classification and Quantitative Evaluation
#### By Jeremy Merrill for Lede 2026


This notebook classifies comments using the OpenAI interface. Right now, it's set up to classify NYT Cooking comments into various categories.

It includes a cost estimation and code for quantitative evaluation (seeing how you did, then adjusting to try to do better).

It can serve as a template for any kind of classification, and you should feel free to copy and reuse it.

In [1]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm.auto import tqdm # makes pretty progress bars
tqdm.pandas()              # makes pretty progress bars in pandas with progress_apply()

# this should be familiar!
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENROUTER_API_KEY_EMR"]

/Users/ericsandwich/.venvs/lede/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'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'

In [2]:
## setting up API key


## importing data
import glob
data_dir = "../csv/"
all_csv_files = [f for f in glob.glob(data_dir + "*.csv") if not f.endswith("_ai_classified.csv")]
data_full_raw = pd.concat([pd.read_csv(f) for f in all_csv_files], ignore_index=True)


SIZE_OF_SAMPLE_FOR_HANDCODING = 50
##
## take a random sample to test our prompts against
text_column_name = "commentBody" # change this to match your own dataset
data_full_raw.dropna(subset=[text_column_name], inplace=True)

# exclude user replies
data_full = data_full_raw[data_full_raw['commentType'] == 'comment']

# if you already did the handcoding, make sure the sample you're using is the right set.
WE_ALREADY_HANDCODED = os.path.exists("handcoded.csv")
if WE_ALREADY_HANDCODED:
  handcoded = pd.read_csv("handcoded.csv")
  data_sample = data_full[data_full.index.isin(handcoded.index)]
else:
  data_sample = data_full.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!

In [3]:
data_sample.head()

,index,author,url,recipeTitle,recipeRating,commentBody,commentType,userID,commentRecommendations,Unnamed: 0,ai_guess,groundtruth
0,0,J. Kenji López-Alt,https://cooking.nytimes.com/recipes/1027100-di...,Diner Burgers Recipe,4(134),Spread thin layer of mayo on buns and toast in...,comment,81048838,69,NaN,NaN,NaN
1,0,J. Kenji López-Alt,https://cooking.nytimes.com/recipes/1027100-di...,Diner Burgers Recipe,4(134),You look at it and think: What kind of human j...,comment,123496796,45,NaN,NaN,NaN
3,0,J. Kenji López-Alt,https://cooking.nytimes.com/recipes/1027100-di...,Diner Burgers Recipe,4(134),I gently hand mix a tablespoon or so of my nic...,comment,73758764,18,NaN,NaN,NaN
5,0,J. Kenji López-Alt,https://cooking.nytimes.com/recipes/1027100-di...,Diner Burgers Recipe,4(134),"I prefer the ketchup, lettuce, onion, and toma...",comment,69047770,18,NaN,NaN,NaN
6,0,J. Kenji López-Alt,https://cooking.nytimes.com/recipes/1027100-di...,Diner Burgers Recipe,4(134),I like to crisp up the buns in the burger fat.,comment,46572225,10,NaN,NaN,NaN


In [4]:

## setting up our connection to an API client for an AI provider
# for Lede, we're using OpenRouter, which connects to most providers
# but I've provided code for connect to other providers directly, if you want to do that in real life

USE_OPENROUTER=True

from anthropic import Anthropic

if USE_OPENROUTER:
  from openrouter import OpenRouter
else:
  from openai import OpenAI
  from mistralai.client import Mistral

if USE_OPENROUTER:
  openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_EMR"))
else:
  openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY_LEDE"))
  mistral_client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY_LEDE"))
anthropic_client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY_LEDE"),) # intentionally created outside of the if; we'll use anthropic_client to count tokens (even if we're using openrouter)


A basic format for a prompt is below. Using the web version of ChatGPT (chatgpt.com) tweak this until you're pretty consistently getting reasonable results.

In [5]:
## prompt
## start with the one YOU tried in the LLM web interface
## don't worry, we'll adjust it later.
## Jeremy's is at the very bottom, if you need to reference it, but please come
##   up with your own.


# Be sure to list the categories (defined below!) with the magic phrase {categories}
prompt_base = """

Here's a comment from NYT Cooking:{comment_text}

Choose the single most specific category that applies from this list:

positive_user_made_as_written
positive_user_describing_changes
thinks_recipe_is_flawed_due_to_flavor
thinks_recipe_is_flawed_due_to_technique_or_equipment
thinks_recipe_is_flawed_due_to_cooking_time
thinks_recipe_is_inauthentic
thinks_ingredients_hard_to_find
user_responding_to_other_user
question
maybe_not_about_the_recipe

Prioritize structural recipe suggestions over general experience.
"""

from enum import Enum, IntEnum

# force the AI to ONLY guess fromn the options we ask you.
# you can change these names! (and perhaps should!)
# - the AI sees the _value_, not the key (it sees "broader_political_issues_except_us_election", not "other_politics")
# - the key is just for you -- a shorthand
# - some AIs have trouble with complicated names, so it's better to stick to lowercase, no special characters
#     (as of early June 2026 -- maybe this will be fixed)
class CommentOptions(str, Enum):
    positive_user_made_as_written = "positive_user_made_as_written"
    positive_user_describing_changes = "positive_user_describing_changes"
    thinks_recipe_is_flawed_due_to_flavor = "thinks_recipe_is_flawed_due_to_flavor"
    thinks_recipe_is_flawed_due_to_technique_or_equipment = "thinks_recipe_is_flawed_due_to_technique_or_equipment"
    thinks_recipe_is_flawed_due_to_cooking_time = "thinks_recipe_is_flawed_due_to_cooking_time"
    thinks_recipe_is_inauthentic = "thinks_recipe_is_inauthentic"
    thinks_ingredients_hard_to_find = "thinks_ingredients_hard_to_find"
    user_responding_to_other_user = "user_responding_to_other_user"
    question = "question"
    maybe_not_about_the_recipe = "maybe_not_about_the_recipe"

# for convenience, make a column in our dataframes with the prompt for each item
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    comment_text=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in CommentOptions])
), axis="columns")
full_data_prompt_column = data_full.apply(lambda row: prompt_base.format(
    comment_text=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in CommentOptions])
), axis="columns")
data_sample_prompt_column.iloc[0]

"\n\nHere's a comment from NYT Cooking:Spread thin layer of mayo on buns and toast in the pan before you add the oil to cook the burgers. You’ll never go back.\n\nChoose the single most specific category that applies from this list:\n\npositive_user_made_as_written\npositive_user_describing_changes\nthinks_recipe_is_flawed_due_to_flavor\nthinks_recipe_is_flawed_due_to_technique_or_equipment\nthinks_recipe_is_flawed_due_to_cooking_time\nthinks_recipe_is_inauthentic\nthinks_ingredients_hard_to_find\nuser_responding_to_other_user\nquestion\nmaybe_not_about_the_recipe\n\nPrioritize structural recipe suggestions over general experience.\n"

In [6]:
## cost estimation

from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4-5": 1 / 1_000_000,
    "claude-sonnet-4-6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [7]:

# IMPORTANT: let's pick a model to use
# it's got to be listed in INPUT_TOKEN_COSTS up above

DEFAULT_MODEL_TO_USE = 'gpt-5.4'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = 'gpt-5.4'

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE


Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4-5
 - claude-sonnet-4-6

The code blocks below will let you estimate costs. If it's too pricey with `gpt-5.4`, consider a cheaper model.

In [8]:
## cost estimation for full dataset
count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $26.01'

In [9]:

## a lot of boilerplate for sending our prompt + comment to the model and getting an answer
from pydantic import BaseModel


class CookingCommentValidOptions(BaseModel):
  classification: CommentOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def anthropic_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    message = anthropic_client.messages.parse(
        max_tokens=1024,
        # system=SYSTEM_PROMPT, # causes trouble with structured outputs, oddly
        messages = [
            {
                "role": "user",
                "content": prompt_including_comment,
            }
        ],
        model=MODEL_TO_USE,
        output_format=CookingCommentValidOptions
    )
    return message.content[0].parsed_output.classification.value if message.content[0].parsed_output else None

def mistral_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    chat_response = mistral_client.chat.parse(
      model=MODEL_TO_USE,
      messages=[
          {
              "role": "system",
              "content": SYSTEM_PROMPT
          },
          {
              "role": "user",
              "content": prompt_including_comment
          },
      ],
      response_format=CookingCommentValidOptions,
      max_tokens=256,
      temperature=0
    )
    return chat_response.choices[0].message.parsed.classification.value if chat_response.choices[0].message.parsed else None

def chatgpt_classify(prompt_including_comment):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_comment,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=CookingCommentValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-4.5-haiku": "anthropic/claude-4.5-haiku",
    "claude-4.6-sonnet": "anthropic/claude-4.6-sonnet"
}
def openrouter_classify(prompt_including_comment):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_comment}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "nyt_comment_classification",
                "strict": True,
                "schema": CookingCommentValidOptions.model_json_schema(),
            },
        },
    )
    return  CookingCommentValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

import time

def classify(prompt_including_comment, max_retries=5):
  "A wrapper for the provider-specific classification functions"
  for attempt in range(max_retries):
    try:
      if USE_OPENROUTER:
        return openrouter_classify(prompt_including_comment)
      else:
        if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
          return mistral_classify(prompt_including_comment)
        elif "claude" in MODEL_TO_USE:
          return anthropic_classify(prompt_including_comment)
        elif "gpt" in MODEL_TO_USE:
          return chatgpt_classify(prompt_including_comment)
        else:
          raise ValueError("Unknown model")
    except Exception as e:
      if attempt == max_retries - 1:
        print(f"Failed after {max_retries} attempts. Error: {e}")
        return None
      sleep_time = 2 ** attempt
      print(f"Error encountered, pausing for {sleep_time} seconds before retrying...")
      time.sleep(sleep_time)


## Doing the whole thing

Once we're ready, and we think we've accounted for most of the mistakes, we're ready to classify the WHOLE THING.

In [ ]:
## classifying full prompt concurrently

import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def robust_classify(prompt, max_retries=10):
    for attempt in range(max_retries):
        try:
            return classify(prompt)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"Final failure after {max_retries} attempts: {e}")
                return None
            time.sleep(3 ** attempt)

def robust_classify_with_progress(prompt):
    res = robust_classify(prompt)
    if 'pbar' in globals():
        pbar.update(1)
    return res

def process_csv(filepath):
    df = pd.read_csv(filepath)
    df.dropna(subset=[text_column_name], inplace=True)
    df = df[df['commentType'] == 'comment'].copy()
    
    if len(df) == 0:
        return filepath
    
    df['prompt'] = df.apply(lambda row: prompt_base.format(
        comment_text=row[text_column_name],
        categories=", ".join(['"{}"'.format(opt.value) for opt in CommentOptions])
    ), axis="columns")
    
    df['ai_guess'] = df['prompt'].apply(robust_classify_with_progress)
    
    basename = os.path.basename(filepath)
    name, ext = os.path.splitext(basename)
    df.to_csv(os.path.join(data_dir, f"{name}_ai_classified.csv"), index=False)
    return filepath

print(f"Processing {len(data_full)} total rows across {len(all_csv_files)} files concurrently...")
pbar = tqdm(total=len(data_full), desc="Total Rows Processed")

with ThreadPoolExecutor(max_workers=18) as executor:
    futures = [executor.submit(process_csv, filepath) for filepath in all_csv_files]
    for future in as_completed(futures):
        filepath = future.result()
pbar.close()
print("Done!")

Processing 60363 total rows across 18 files concurrently...


Total Rows Processed:   0%|          | 105/60363 [00:20<4:20:15,  3.86it/s]